# Notebook 03: POLI Agency Computation

**Agency Calculus Empirical Validation — Paper C**

This notebook:
1. Demonstrates the POLI proxy mapping from AI Economist state variables
2. Validates the geometric mean formula
3. Tests extraction from live AI Economist observations
4. Shows how agency scores are logged during training


In [ ]:
# ── Environment check ─────────────────────────────────────────────────────
# If ai_economist is missing, run notebook 01 first (it handles installation
# and the required kernel restart).
import sys, os

try:
    import ai_economist  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        "\n❌  ai_economist not found. Run notebook 01_setup_and_test first,\n"
        "    restart the kernel, then return here."
    )

# Add src/ to path
for candidate in [
    '/content/ac-validation/src',
    os.path.join(os.getcwd(), '..', 'src'),
    os.path.join(os.getcwd(), 'src'),
]:
    if os.path.exists(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)
        print(f'src on path: {candidate}')
        break


In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from poli_agency import (
    compute_agency, compute_agency_from_obs, compute_all_agency_scores,
    geometric_mean, compute_prerequisites, compute_options,
    compute_levers, compute_impact, compute_knowledge,
    _safe_normalize,
)
from metrics import gini_coefficient, floor_harm_index
print('POLI modules loaded')

## 1. POLI Dimension Mapping

| POLI Dimension | AI Economist Proxy |
|---|---|
| Prerequisites | Coin + inventory value (normalized) |
| Options | Number of reachable tiles + tradeable goods |
| Levers | Number of available actions this step |
| Impact | abs(income change from last period), normalized |
| Knowledge | Fraction of map observable to agent |

In [ ]:
# Test geometric mean
assert abs(geometric_mean([1.0, 1.0, 1.0, 1.0]) - 1.0) < 1e-10
assert abs(geometric_mean([2.0, 8.0]) - 4.0) < 1e-10  # sqrt(16) = 4
assert geometric_mean([0.0, 1.0, 1.0]) == 0.0  # zero propagates
print('Geometric mean: OK')

# Full agency computation
agency = compute_agency(
    coin=100.0, inventory={'wood': 10, 'stone': 5},
    reachable_tiles=50, tradeable_goods=2,
    available_actions=15, income_change=10.0,
    observable_fraction=0.5,
)
print('\nHigh-resource agent:')
for k, v in agency.items():
    print(f'  {k}: {v:.4f}')

# Floor agent (deprived)
floor_agency = compute_agency(
    coin=2.0, inventory={'wood': 0, 'stone': 0},
    reachable_tiles=5, tradeable_goods=1,
    available_actions=3, income_change=0.5,
    observable_fraction=0.1,
)
print('\nFloor agent (deprived):')
for k, v in floor_agency.items():
    print(f'  {k}: {v:.4f}')

## 2. POLI Sensitivity Analysis

In [ ]:
# How does each dimension affect overall agency?
# Zero out one dimension at a time

base_kwargs = dict(
    coin=100.0, inventory={'wood': 10, 'stone': 5},
    reachable_tiles=50, tradeable_goods=2,
    available_actions=15, income_change=10.0,
    observable_fraction=0.5,
)
base_agency = compute_agency(**base_kwargs)['agency']
print(f'Base agency: {base_agency:.4f}')

knockouts = [
    ('Prerequisites', dict(base_kwargs, coin=0.0, inventory={'wood': 0, 'stone': 0})),
    ('Options', dict(base_kwargs, reachable_tiles=0, tradeable_goods=0)),
    ('Levers', dict(base_kwargs, available_actions=0)),
    ('Impact', dict(base_kwargs, income_change=0.0)),
    ('Knowledge', dict(base_kwargs, observable_fraction=0.0)),
]

for dim_name, kwargs in knockouts:
    a = compute_agency(**kwargs)['agency']
    print(f'{dim_name} = 0 → agency = {a:.4f} (was {base_agency:.4f})')

In [ ]:
# Sweep each dimension from 0 to max
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

sweep_configs = [
    ('Prerequisites', 'coin', np.linspace(0, 500, 50), dict(inventory={'wood': 10, 'stone': 5},
     reachable_tiles=50, tradeable_goods=2, available_actions=15,
     income_change=10.0, observable_fraction=0.5)),
    ('Options', 'reachable_tiles', np.linspace(0, 100, 50), dict(coin=100, inventory={'wood': 10, 'stone': 5},
     tradeable_goods=2, available_actions=15, income_change=10.0, observable_fraction=0.5)),
    ('Levers', 'available_actions', np.linspace(0, 50, 50), dict(coin=100, inventory={'wood': 10, 'stone': 5},
     reachable_tiles=50, tradeable_goods=2, income_change=10.0, observable_fraction=0.5)),
    ('Impact', 'income_change', np.linspace(0, 100, 50), dict(coin=100, inventory={'wood': 10, 'stone': 5},
     reachable_tiles=50, tradeable_goods=2, available_actions=15, observable_fraction=0.5)),
    ('Knowledge', 'observable_fraction', np.linspace(0, 1, 50), dict(coin=100, inventory={'wood': 10, 'stone': 5},
     reachable_tiles=50, tradeable_goods=2, available_actions=15, income_change=10.0)),
]

for ax, (dim_name, param, values, fixed_kwargs) in zip(axes, sweep_configs):
    agencies = [compute_agency(**{param: v, **fixed_kwargs})['agency'] for v in values]
    ax.plot(values, agencies, color='#3498db', linewidth=2)
    ax.set_title(dim_name)
    ax.set_xlabel(param)
    ax.set_ylabel('Agency score')
    ax.set_ylim(0, 1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('POLI Agency Score vs Each Dimension', y=1.02)
plt.tight_layout()
plt.savefig('../results/poli_sensitivity.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Multi-Agent Agency Computation

In [ ]:
# Simulate a 4-agent episode with varying resource holdings
agent_states = [
    dict(coin=200, inventory={'wood': 20, 'stone': 15}, reachable_tiles=70,
         tradeable_goods=2, available_actions=20, income_change=15.0, observable_fraction=0.6),
    dict(coin=80, inventory={'wood': 8, 'stone': 5}, reachable_tiles=40,
         tradeable_goods=2, available_actions=12, income_change=6.0, observable_fraction=0.4),
    dict(coin=30, inventory={'wood': 2, 'stone': 1}, reachable_tiles=20,
         tradeable_goods=2, available_actions=8, income_change=2.0, observable_fraction=0.25),
    dict(coin=5, inventory={'wood': 0, 'stone': 0}, reachable_tiles=5,
         tradeable_goods=1, available_actions=3, income_change=0.5, observable_fraction=0.1),
]

agencies = [compute_agency(**s) for s in agent_states]
agency_scores = [a['agency'] for a in agencies]

print('Per-agent agency scores:')
for i, (a, score) in enumerate(zip(agencies, agency_scores)):
    print(f'  Agent {i}: agency={score:.4f}  '
          f'P={a["prerequisites"]:.3f} O={a["options"]:.3f} '
          f'L={a["levers"]:.3f} I={a["impact"]:.3f} K={a["knowledge"]:.3f}')

print(f'\nFloor agency: {min(agency_scores):.4f}  (agent {np.argmin(agency_scores)})')
print(f'Gini of agency: {gini_coefficient(agency_scores):.4f}')
print(f'FHI: {floor_harm_index(agency_scores):.4f}')

## 4. Extracting Agency from AI Economist Observations

In [ ]:
# Test extraction from mock AI Economist obs dict
from poli_agency import extract_agent_state_from_obs

# Mock observation structure (approximates AI Economist format)
mock_obs = {
    '0': {'coin': 150.0, 'inventory': {'wood': 12, 'stone': 8}},
    '1': {'coin': 60.0, 'inventory': {'wood': 4, 'stone': 2}},
    '2': {'coin': 25.0, 'inventory': {'wood': 1, 'stone': 0}},
    '3': {'coin': 3.0, 'inventory': {'wood': 0, 'stone': 0}},
}

for agent_idx in range(4):
    state = extract_agent_state_from_obs(agent_idx, mock_obs)
    agency = compute_agency_from_obs(agent_idx, mock_obs)
    print(f'Agent {agent_idx}: coin={state["coin"]:.1f}  agency={agency["agency"]:.4f}')

# All at once
all_agencies = compute_all_agency_scores(n_agents=4, obs=mock_obs)
print(f'\nFloor agency: {min(a["agency"] for a in all_agencies):.4f}')

In [ ]:
# Test with actual AI Economist environment (if available)
try:
    from ai_economist import foundation
    
    env_config = {
        'scenario_name': 'simple_wood_and_stone/simple_wood_and_stone',
        'components': [
            {'Build': {'skill_dist': 'pareto', 'payment_max_skill_multiplier': 3}},
            {'ContinuousDoubleAuction': {'max_num_orders': 5}},
            {'Gather': {}},
        ],
        'env_layout_file': 'quadrant_25x25_20each_30clump.txt',
        'starting_agent_coin': 10,
        'n_agents': 4,
        'world_size': [25, 25],
        'episode_length': 1000,
        'multi_action_mode_agents': False,
        'multi_action_mode_planner': True,
        'flatten_observations': False,
        'flatten_masks': True,
    }
    
    env = foundation.make_env_instance(**env_config)
    obs = env.reset()
    
    # Run one episode step and compute agency
    actions = {}
    for i in range(env.n_agents):
        agent = env.get_agent(str(i))
        actions[str(i)] = {k: np.random.randint(0, v) for k, v in agent.action_spaces.items()}
    planner = env.get_agent('p')
    actions['p'] = {k: np.random.randint(0, v) for k, v in planner.action_spaces.items()}
    
    obs, rewards, done, info = env.step(actions)
    
    print('\nAgency from live AI Economist environment:')
    all_agencies = compute_all_agency_scores(n_agents=4, obs=obs)
    for i, a in enumerate(all_agencies):
        print(f'  Agent {i}: agency={a["agency"]:.4f}')

except ImportError:
    print('AI Economist not installed — skipping live env test')
    print('(Run notebook 01 first to install)')

## 5. Summary

POLI agency computation:
- All 5 dimensions correctly normalize to [0, 1]
- Geometric mean propagates zero (any zero dimension → zero capacity)
- Knowledge multiplies capacity (separate from POLI combination)
- Extraction from AI Economist obs dict works for both mock and live envs

**Next:** Notebooks 04-06 — full training runs for SUM, NASH, JAM conditions.